In [1]:
import os, sys, shutil
from pathlib import Path
try:
    from google.colab import userdata, drive
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
except Exception:
    import getpass
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY: ')
if not os.path.exists('/content/mmai'):
    !git clone https://github.com/michaelyserrano/mmai.git /content/mmai
    !pip install -q -r /content/mmai/project/requirements.txt
%cd /content/mmai
sys.path.insert(0, 'project')
drive.mount('/content/drive')
DRIVE_TB = Path('/content/drive/MyDrive/Colab Notebooks/toolbench_data')
LOCAL_TB = Path('/content/toolbench_data')
if not LOCAL_TB.exists():
    print('Copying toolbench to local disk (first time, slow)...')
    !cp -r "{DRIVE_TB}" "{LOCAL_TB}"
else:
    for fname in ['toolllama_G123_dfs_eval.json', 'toolllama_G123_dfs_train.json']:
        if not (LOCAL_TB / fname).exists():
            shutil.copy(DRIVE_TB / fname, LOCAL_TB / fname)
            print(f'Copied missing {fname}')
TOOLBENCH_DIR = LOCAL_TB

/content/mmai
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copied missing toolllama_G123_dfs_eval.json
Copied missing toolllama_G123_dfs_train.json


In [2]:
# Restore pre-computed outputs from Drive if not already on local disk
import shutil
from pathlib import Path
drive_out = Path('/content/drive/MyDrive/Colab Notebooks/mmai/project')
for fname in ['corpus_embeddings.npy', 'api_names.json']:
    dst = Path('project') / fname
    src = drive_out / fname
    if not dst.exists() and src.exists():
        shutil.copy(src, dst)
        print(f'Restored {fname} from Drive')
    elif dst.exists():
        print(f'{fname} already on local disk')
    else:
        print(f'WARNING: {fname} not found — run notebook 01 first')

corpus_embeddings.npy already on local disk
api_names.json already on local disk


In [3]:
import numpy as np, json
from pathlib import Path
from data.load_toolbench import load_api_corpus, load_eval_examples
from models.embeddings import get_embeddings
from retrieval.retriever import build_faiss_index, retrieve_top_k
from evaluation.metrics import evaluate_batch

corpus = load_api_corpus(Path(TOOLBENCH_DIR) / "toolenv" / "tools")
corpus_matrix = np.load("project/corpus_embeddings.npy")
with open("project/api_names.json") as f:
    api_names = json.load(f)
name_to_idx = {name: i for i, name in enumerate(api_names)}
# api_names contains action_names (e.g. racecards_for_greyhound_racing_uk)
evals = load_eval_examples(TOOLBENCH_DIR / "toolllama_G123_dfs_eval.json")
print(f"Corpus: {len(corpus)} APIs | Eval: {len(evals)} examples")

Corpus: 8724 APIs | Eval: 747 examples


In [4]:
index = build_faiss_index(corpus_matrix)
print(f"\u2713 FAISS index ready: {index.ntotal} vectors")

✓ FAISS index ready: 2733 vectors


In [5]:
from tqdm import tqdm
queries = [e["user_query"] for e in evals]
query_embeddings = get_embeddings(queries)
print(f"\u2713 Query embeddings: {query_embeddings.shape}")

✓ Query embeddings: (747, 1536)


In [6]:
K = 10
all_retrieved = []
all_gt_indices = []

for i, ex in enumerate(evals):
    top_k_indices = retrieve_top_k(query_embeddings[i], index, k=K)
    all_retrieved.append(top_k_indices)
    gt_indices = [name_to_idx[name] for name in ex["ground_truth_apis"] if name in name_to_idx]
    all_gt_indices.append(gt_indices)

results = evaluate_batch(all_retrieved, all_gt_indices, ks=[1, 5, 10])
print("\n=== Baseline Results (Full Corpus / Random Negatives) ===")
for metric, score in results.items():
    print(f"  {metric}: {score:.4f}")


=== Baseline Results (Full Corpus / Random Negatives) ===
  recall@1: 1.0000
  recall@5: 1.0000
  recall@10: 1.0000
  mrr: 0.0000


In [7]:
import json, shutil
from pathlib import Path
with open('project/results_baseline.json', 'w') as f:
    json.dump({'condition': 'random_full_corpus', 'results': results}, f, indent=2)
drive_out = Path('/content/drive/MyDrive/Colab Notebooks/mmai/project')
drive_out.mkdir(parents=True, exist_ok=True)
shutil.copy('project/results_baseline.json', drive_out / 'results_baseline.json')
print(f'Saved to Drive: {drive_out}')

Saved to Drive: /content/drive/MyDrive/Colab Notebooks/mmai/project
